In [ ]:
import asyncio
import random
import datetime
import redis.asyncio as redis
import nest_asyncio

nest_asyncio.apply()

num_test_streams = 3
pub_freq = 1
stream_max_len = 100

async def publish_test_data_for_stream(stream_index, redis_client):
    last_price = 100.0  # Starting price
    stream_key = f"test_{stream_index}"  # Using test_1, test_2, etc.

    while True:
        # Simulate large swings by adding more volatility
        change = random.uniform(-5, 5)  # Increased fluctuation range
        last_price = max(10, last_price + change)  # Keep price above zero

        # Force RSI boundary conditions sometimes
        if random.random() < 0.1:  
            last_price *= random.choice([0.85, 1.15])  # Big jumps 15% up or down

        # Create fake OHLC data
        data = {
            "symbol": "TEST",
            "timestamp": datetime.datetime.now(datetime.UTC).isoformat(),
            "open": round(last_price - random.uniform(0.5, 2), 2),
            "high": round(last_price + random.uniform(0.5, 2), 2),
            "low": round(last_price - random.uniform(1, 3), 2),
            "close": round(last_price, 2),
            "volume": random.randint(100, 1000),
            "trade_count": random.randint(10, 50),
            "vwap": round(last_price + random.uniform(-1, 1), 2),
        }

        # Push data to Redis stream
        await redis_client.xadd(stream_key, data, maxlen=stream_max_len)
        print(f"Pushed to {stream_key}: {data}")

        await asyncio.sleep(pub_freq)  # Adjust frequency if needed

async def publish_test_data(num_streams=1):
    redis_client = redis.Redis(host='localhost', port=6379, decode_responses=True)

    # Create a list of tasks to run multiple streams concurrently
    tasks = []
    for stream_index in range(1, num_streams + 1):
        task = asyncio.create_task(publish_test_data_for_stream(stream_index, redis_client))
        tasks.append(task)

    # Run all the tasks concurrently
    await asyncio.gather(*tasks)


await publish_test_data(num_streams=num_test_streams)

Pushed to test_2: {'symbol': 'TEST', 'timestamp': '2025-03-24T13:56:21.654490+00:00', 'open': 101.81, 'high': 104.14, 'low': 101.04, 'close': 103.28, 'volume': 108, 'trade_count': 29, 'vwap': 103.2}
Pushed to test_3: {'symbol': 'TEST', 'timestamp': '2025-03-24T13:56:21.656000+00:00', 'open': 119.46, 'high': 121.12, 'low': 118.49, 'close': 120.28, 'volume': 409, 'trade_count': 32, 'vwap': 119.41}
Pushed to test_1: {'symbol': 'TEST', 'timestamp': '2025-03-24T13:56:21.653491+00:00', 'open': 98.85, 'high': 101.26, 'low': 98.11, 'close': 100.54, 'volume': 875, 'trade_count': 48, 'vwap': 100.59}
Pushed to test_3: {'symbol': 'TEST', 'timestamp': '2025-03-24T13:56:22.671181+00:00', 'open': 123.72, 'high': 126.98, 'low': 122.39, 'close': 125.08, 'volume': 192, 'trade_count': 39, 'vwap': 126.07}
Pushed to test_1: {'symbol': 'TEST', 'timestamp': '2025-03-24T13:56:22.671181+00:00', 'open': 101.84, 'high': 105.28, 'low': 101.93, 'close': 103.3, 'volume': 943, 'trade_count': 33, 'vwap': 103.43}
Push